# CP322 - Assignemnt 3 (IMBD)

**Author:** Jay Patel  
**Course:** CP322  
**Notebook created:** 2025-11-15 20:46

This notebook follows the assignment brief exactly and is structured into the required sections:

1. Data Loading & Preprocessing
2. CNN (PyTorch)
3. LSTM (PyTorch)
4. BERT Fine-Tuning (Hugging Face) (Bonus)


## Setup

In [1]:
%pip install torch torchvision torchaudio transformers --quiet

import os, re, string, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

plt.rcParams.update({"figure.figsize": (8,4), "axes.grid": True})
SEED = 19
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Note: you may need to restart the kernel to use updated packages.


# 1) Data Loading & Preprocessing 

In [2]:
# 1.1 Load IMDB (Kaggle 50k: columns 'review','sentiment')
CSV = "IMDB Dataset.csv"  # put the Kaggle CSV beside this notebook
df = pd.read_csv(CSV)

# 1.2 Map labels to {neg:0,pos:1}
label_map = {"negative":0, "positive":1}
df["label"] = df["sentiment"].str.lower().map(label_map).astype(int)

# 1.3 Clean text: lowercase, remove HTML, keep letters/numbers/spaces, drop stopwords
_re_html = re.compile(r"<.*?>")
_stop = set(ENGLISH_STOP_WORDS)

def clean_text(s: str) -> str:
    s = s.lower()
    s = _re_html.sub(" ", s)
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    tokens = [t for t in s.split() if t not in _stop]
    return " ".join(tokens)

df["text"] = df["review"].astype(str).apply(clean_text)

# 1.4 Split 70/15/15 (train/val/test), stratified
train_df, temp_df = train_test_split(df[["text","label"]], test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

len(train_df), len(val_df), len(test_df)

# 1.5 Tokenization & Vocabulary (train-only), then encoder
from collections import Counter
PAD_IDX, UNK_IDX = 0, 1
MAX_LEN = 100

def build_vocab(texts, min_freq=2):
    cnt = Counter()
    for s in texts:
        cnt.update(s.split())              # whitespace tokens (after cleaning)
    itos = ["<PAD>", "<UNK>"] + [tok for tok, c in cnt.items() if c >= min_freq]
    stoi = {tok: i for i, tok in enumerate(itos)}
    return stoi, itos

stoi, itos = build_vocab(train_df["text"].tolist(), min_freq=2)
print("Vocab size:", len(itos), "| Examples:", itos[:10])

def encode(text: str, stoi=stoi, max_len=MAX_LEN):
    ids = [stoi.get(tok, UNK_IDX) for tok in text.split()]
    if len(ids) < max_len:
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return np.array(ids, dtype=np.int64)

# quick sanity peek
x_sample = encode(train_df["text"].iloc[0])
print("Encoded length:", len(x_sample), "| first 12 ids:", x_sample[:12])

Vocab size: 63416 | Examples: ['<PAD>', '<UNK>', 'attractive', 'capable', 'cast', 'lost', 'deadly', 'boring', 'rehash', 'slasher']
Encoded length: 100 | first 12 ids: [ 2  3  4  5  6  7  8  9 10 11 12 13]


# 2) CNN (PyTorch)

In [3]:
# Create PyTorch Dataset and DataLoaders
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, encode_fn):
        self.texts = texts
        self.labels = labels
        self.encode_fn = encode_fn
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        label = self.labels.iloc[idx]
        encoded = self.encode_fn(text)
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# Create datasets
train_dataset = IMDBDataset(train_df["text"], train_df["label"], encode)
val_dataset = IMDBDataset(val_df["text"], val_df["label"], encode)
test_dataset = IMDBDataset(test_df["text"], test_df["label"], encode)

# Create data loaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Define embedding dimension
EMB_DIM = 128


In [4]:
# --- 2) CNN (PyTorch) — per spec (emb=128, Conv1d 100@k=3, ReLU, MaxPool k=2, FC->2) ---

import torch
import torch.nn as nn
from torch.utils.data import DataLoader


class TextCNN(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 128, num_filters: int = 100, kernel_size: int = 3):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)  # <PAD>=0, <UNK>=1 handled upstream
        self.conv = nn.Conv1d(in_channels=emb_dim, out_channels=num_filters, kernel_size=kernel_size, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)  # halves the sequence length
        self.fc   = nn.Linear(num_filters * (MAX_LEN // 2), 2)  # binary logits

    def forward(self, x):                 # x: [B, L]
        e = self.emb(x)                   # [B, L, E]
        z = e.transpose(1, 2)             # [B, E, L]
        z = self.pool(self.relu(self.conv(z)))  # [B, F, L/2]
        z = z.reshape(z.size(0), -1)      # flatten
        return self.fc(z)                 # [B, 2] logits

def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, total_correct, total_n = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()
    with torch.set_grad_enabled(train_mode):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_n += xb.size(0)
    avg_loss = total_loss / max(1, total_n)
    avg_acc  = total_correct / max(1, total_n)
    return avg_loss, avg_acc

cnn = TextCNN(vocab_size=len(itos), emb_dim=EMB_DIM).to(DEVICE)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)

for epoch in range(1, 4):
    tr_loss, tr_acc = run_epoch(cnn, train_loader, optimizer)
    # eval
    va_loss, va_acc = run_epoch(cnn, val_loader, optimizer=None)  # eval mode inside
    print(f"[CNN] Epoch {epoch:02d} | train loss {tr_loss:.4f} | val acc {va_acc:.4f}")



[CNN] Epoch 01 | train loss 0.5220 | val acc 0.8105
[CNN] Epoch 02 | train loss 0.2652 | val acc 0.8304
[CNN] Epoch 03 | train loss 0.1055 | val acc 0.8324


# 3) LSTM (PyTorch)

In [5]:
# --- 3) LSTM (PyTorch) — emb=128, hidden=128, CE, Adam 1e-3, 3 epochs, val acc each epoch ---

# constants (in case not already defined above)
if 'EMB_DIM' not in globals(): EMB_DIM = 128
if 'PAD_IDX' not in globals(): PAD_IDX = 0
if 'DEVICE'  not in globals():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TextLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128, bidir=False):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(input_size=emb_dim, hidden_size=hidden,
                            num_layers=1, batch_first=True, bidirectional=bidir)
        out_dim = hidden * (2 if bidir else 1)
        self.fc   = nn.Linear(out_dim, 2)  # binary logits

    def forward(self, x):                 # x: [B, L]
        e = self.emb(x)                   # [B, L, E]
        _, (h_n, _) = self.lstm(e)        # h_n: [layers*(2 if bi), B, H]
        h_last = h_n[-1]                  # [B, H] (last layer)
        return self.fc(h_last)            # [B, 2]

def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0; total_correct = 0; total_n = 0
    crit = nn.CrossEntropyLoss()
    with torch.set_grad_enabled(train_mode):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = crit(logits, yb)
            if train_mode:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss   += loss.item() * xb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_n      += xb.size(0)
    return total_loss / max(1,total_n), total_correct / max(1,total_n)

# train for 3 epochs and report val accuracy each epoch
lstm = TextLSTM(vocab_size=len(itos), emb_dim=EMB_DIM, hidden=128, bidir=False).to(DEVICE)
opt_l = torch.optim.Adam(lstm.parameters(), lr=1e-3)

for ep in range(1, 4):
    tr_loss, tr_acc = run_epoch(lstm, train_loader, opt_l)
    va_loss, va_acc = run_epoch(lstm, val_loader,  None)
    print(f"[LSTM] Epoch {ep:02d} | train loss {tr_loss:.4f} | val acc {va_acc:.4f}")


[LSTM] Epoch 01 | train loss 0.6801 | val acc 0.6317
[LSTM] Epoch 02 | train loss 0.6219 | val acc 0.5621
[LSTM] Epoch 03 | train loss 0.6000 | val acc 0.7767


# 4) BERT Fine-Tuning (Higging Face) (Bonus)

In [ ]:
# --- 4) BONUS: BERT Fine-Tuning (fixed AdamW import) ---

# !pip install transformers --quiet

import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW                      # <-- use PyTorch AdamW
from transformers import (
    BertTokenizerFast, BertForSequenceClassification,
    get_linear_schedule_with_warmup               # still from transformers
)

# device / constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN_BERT = 128
BATCH_BERT   = 8
EPOCHS_BERT  = 2
LR_BERT      = 2e-5

# 1) Tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def to_bert_dataset(df_frame):
    enc = tokenizer(
        df_frame["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN_BERT,
        return_tensors="pt"
    )
    labels = torch.tensor(df_frame["label"].astype(int).values)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

# 2) Data (uses your 70/15/15 splits)
train_ds_b = to_bert_dataset(train_df)
val_ds_b   = to_bert_dataset(val_df)
test_ds_b  = to_bert_dataset(test_df)

train_loader_b = DataLoader(train_ds_b, batch_size=BATCH_BERT, shuffle=True)
val_loader_b   = DataLoader(val_ds_b,   batch_size=BATCH_BERT)
test_loader_b  = DataLoader(test_ds_b,  batch_size=BATCH_BERT)

# 3) Model
model_b = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(DEVICE)

# 4) Optimizer (PyTorch AdamW) + scheduler
optimizer_b = AdamW(model_b.parameters(), lr=LR_BERT)
num_training_steps = EPOCHS_BERT * len(train_loader_b)
scheduler_b = get_linear_schedule_with_warmup(optimizer_b, num_warmup_steps=0, num_training_steps=num_training_steps)

@torch.no_grad()
def eval_bert(model, loader):
    model.eval(); correct=n=0
    for ids, mask, y in loader:
        ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
        out = model(input_ids=ids, attention_mask=mask, labels=y)
        pred = out.logits.argmax(1)
        correct += (pred==y).sum().item(); n += y.size(0)
    return correct/max(1,n)

# 5) Train 2 epochs
for ep in range(1, EPOCHS_BERT+1):
    model_b.train(); running=seen=0
    for ids, mask, y in train_loader_b:
        ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
        out = model_b(input_ids=ids, attention_mask=mask, labels=y)
        loss = out.loss
        optimizer_b.zero_grad(); loss.backward(); optimizer_b.step(); scheduler_b.step()
        running += loss.item()*y.size(0); seen += y.size(0)
    val_acc = eval_bert(model_b, val_loader_b)
    print(f"[BERT] Epoch {ep:02d} | train loss {running/max(1,seen):.4f} | val acc {val_acc:.4f}")

# 6) Test
test_acc_b = eval_bert(model_b, test_loader_b)
print(f"[TEST] BERT acc = {test_acc_b:.4f}")


Note: you may need to restart the kernel to use updated packages.


ImportError: cannot import name 'AdamW' from 'transformers' (/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/__init__.py)